In [23]:
# Impotação das bibliotecas que serão utilizadas e busca dos dados no csv
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

caminho_dados = 'https://raw.githubusercontent.com/AnaRaquelCafe/POSTECH_AI_SCIENTIST/refs/heads/main/Base%20de%20dados%20Tech%20Challenge/desafio_nps_fase_1.csv'

df = pd.read_csv(caminho_dados)

df.head(10)

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9
5,6,35,Sudeste,75,50006,568.76,6,36.58,3,4,5,41.82,2,2,3,0.9,0,5,1.5
6,7,37,Sudeste,68,50007,41.29,3,99.62,6,8,3,35.83,3,3,4,1.4,0,6,0.6
7,8,60,Sul,37,50008,428.76,4,29.54,10,11,5,44.50,1,0,2,0.0,0,2,4.1
8,9,40,Sul,60,50009,121.56,3,91.95,6,6,3,24.88,2,1,9,6.2,0,3,0.8
9,10,51,Sudeste,70,50010,411.01,6,37.47,3,9,2,30.59,1,0,7,2.7,0,2,4.2


In [24]:
# Assim como fizemos no EDA, vamos renomear as colunas para o nosso idioma, facilitando o entendimento e manipulação
df = df.rename(
    columns={
        'nps_score': 'nota_nps',
        'delivery_time_days': 'tempo_entrega_dias',
        'delivery_delay_days': 'dias_atraso_entrega',
        'delivery_attempts': 'tentativas_entrega',
        'customer_service_contacts': 'contatos_sac',
        'resolution_time_days': 'tempo_resolucao_dias',
        'complaints_count': 'qtd_reclamacoes',
        'customer_age': 'idade_cliente',
        'customer_tenure_months': 'tempo_relacionamento_cliente',
        'order_value': 'valor_pedido',
        'items_quantity': 'quantidade_itens',
        'discount_value': 'desconto_pedido',
        'freight_value': 'valor_frete',
        'csat_internal_score': 'nota_csat',
        'customer_region': 'regiao_cliente',
        'order_id': 'id_pedido'
    }
)


MODELO DE REGRESSÃO

In [25]:
# Definindo a variável alvo
Y = df['nota_nps']

# Selecionando apenas as variáveis de entrada operacionais relevantes conforme o EDA
colunas_operacionais = [
    'dias_atraso_entrega',
    'qtd_reclamacoes',
    'contatos_sac',
    'tempo_resolucao_dias'
]

X = df[colunas_operacionais]

# Adicionando a constante (intercepto) à matriz de variáveis explicativas.
# O statsmodels exige essa etapa para calcular o valor base da nota do NPS 
# quando todas as variáveis forem iguais a zero.
X_com_const = sm.add_constant(X)

# Verificando as 5 primeiras linhas da matriz X pronta
X_com_const.head()

,const,dias_atraso_entrega,qtd_reclamacoes,contatos_sac,tempo_resolucao_dias
0,1.0,2,3,0,4
1,1.0,4,3,0,10
2,1.0,1,7,4,5
3,1.0,2,4,1,11
4,1.0,1,3,1,0


In [26]:
# Dividindo a base: 80% para treino e 20% para teste
X_train, X_test, y_train, y_test = train_test_split(
    X_com_const, 
    Y, 
    test_size=0.20, 
    random_state=42
)

# Conferindo o tamanho de cada base
print(f"Clientes na base de Treino: {len(X_train)}")
print(f"Clientes na base de Teste: {len(X_test)}")

Clientes na base de Treino: 2000
Clientes na base de Teste: 500


In [27]:
# Ajustando o modelo de Regressão Linear por Mínimos Quadrados Ordinários (OLS).
# O objetivo é quantificar a penalidade exata que cada dia de atraso, reclamação, contatos com 
# o SAC e tempo de resolução de problemas geram no NPS
modelo_nps = sm.OLS(y_train, X_train).fit()

# Exibindo o sumário estatístico completo para validar p-valor, R² e coeficientes de regressão
print(modelo_nps.summary())

                            OLS Regression Results                            
Dep. Variable:               nota_nps   R-squared:                       0.556
Model:                            OLS   Adj. R-squared:                  0.555
Method:                 Least Squares   F-statistic:                     623.5
Date:                Sun, 30 Aug 2026   Prob (F-statistic):               0.00
Time:                        16:01:33   Log-Likelihood:                -3866.4
No. Observations:                2000   AIC:                             7743.
Df Residuals:                    1995   BIC:                             7771.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    9.3426 

In [28]:
# 1. Qualidade do Ajuste (R² = 0,556):
#    O modelo consegue explicar 55,6% de toda a variação da nota do NPS utilizando apenas as 4 variáveis operacionais.
#
# 2. Significância Estatística (P>|t| = 0,000):
#    Todas as variáveis apresentaram p-valor de 0,000. 
#    Isso rejeita a hipótese nula, provando que o impacto de cada
#    variável sobre a queda do NPS é real e estatisticamente comprovado.
#
# 3. Equação do NPS (Impacto Real no Negócio):
#    NPS Esperado = 9,34 - (0,94 * dias_atraso) - (0,39 * qtd_reclamacoes) - (0,33 * contatos_sac) - (0,15 * tempo_resolucao)
#
#    - Base (Intercepto = 9,34): Sem nenhum problema logístico ou de SAC, a nota média estimada do cliente é 9,3.
#    - Maior Ofensor (Dias de Atraso): Cada 1 dia de atraso reduz a nota do NPS em quase 1 ponto inteiro (-0,94).
#    - Efeito SAC: Cada nova reclamação custa -0,39 na nota, enquanto cada contato adicional reduz -0,33.

In [29]:
# Realizando as predições no conjunto de TESTE
y_pred = modelo_nps.predict(X_test)

# Calculando as métricas de erro MAE e RMSE
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Erro Médio Absoluto (MAE): {mae:.2f} pontos no NPS")
print(f"Erro Quadrático Médio (MSE): {mse:.2f} pontos no NPS")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f} pontos no NPS")

Erro Médio Absoluto (MAE): 1.32 pontos no NPS
Erro Quadrático Médio (MSE): 2.80 pontos no NPS
Raiz do Erro Quadrático Médio (RMSE): 1.67 pontos no NPS


In [30]:
# 1. MAE (Erro Médio Absoluto = 1,32):
#    Em média, as previsões do modelo erram a nota do NPS por cerca de 1,32 ponto (para mais ou para menos).
#
# 2. MSE (Erro Quadrático Médio = 2,80):
#    Métrica que penaliza erros maiores ao elevá-los ao quadrado. Serve de base direta para o cálculo do RMSE.
#
# 3. RMSE (Raiz do Erro Quadrático Médio = 1.67):
#    Como o RMSE (1,67) ficou próximo do MAE (1,32), isso confirma que o modelo não está cometendo grandes
#    erros atípicos e mantém uma margem de erro estável para prever a insatisfação do cliente.

MODELO DE CLASSIFICAÇÃO

In [31]:
# Criando a coluna 'fl_detrator': 1 se nps_score < 7, caso contrário 0
df['fl_detrator'] = (df['nota_nps'] < 7.0).astype(int)

# Verificando a distribuição da nova variável alvo
print(df['fl_detrator'].value_counts())
print(df['fl_detrator'].value_counts(normalize=True) * 100)

fl_detrator
1    2109
0     391
Name: count, dtype: int64
fl_detrator
1    84.36
0    15.64
Name: proportion, dtype: float64


In [32]:
# Esse nosso modelo agora vai ter um objetivo prático de identificar previamente os clientes 
# com alto risco de se tornarem detratores (fl_detrator = 1) com base nas variáveis operacionais 
# que identificamos como relevantes no nosso EDA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Definindo o novo Y (variável binária) e o X (as 4 variáveis operacionais)
Y_class = df['fl_detrator']
X_class = df[['dias_atraso_entrega', 'qtd_reclamacoes', 'contatos_sac', 'tempo_resolucao_dias']]

# Dividindo novamente em treino (80%) e teste (20%)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, Y_class, test_size=0.20, random_state=42
)

# Criando e treinando o modelo de regressão logística
modelo_logistico = LogisticRegression()
modelo_logistico.fit(X_train_c, y_train_c)

# Fazendo as previsões na base de teste
y_pred_class = modelo_logistico.predict(X_test_c)

# Gerando a Matriz de Confusão
matriz = confusion_matrix(y_test_c, y_pred_class)

# Exibindo os resultados de forma clara
print("=== MATRIZ DE CONFUSÃO ===")
print(matriz)
print("\n=== RELATÓRIO DE DESEMPENHO ===")
print(classification_report(y_test_c, y_pred_class, target_names=['Não-Detrator (0)', 'Detrator (1)']))

=== MATRIZ DE CONFUSÃO ===
[[ 24  49]
 [ 20 407]]

=== RELATÓRIO DE DESEMPENHO ===
                  precision    recall  f1-score   support

Não-Detrator (0)       0.55      0.33      0.41        73
    Detrator (1)       0.89      0.95      0.92       427

        accuracy                           0.86       500
       macro avg       0.72      0.64      0.67       500
    weighted avg       0.84      0.86      0.85       500



In [33]:
# 1. Desempenho na Classe Alvo (Detrator - Classe 1):
#    - Recall (0,95): O modelo é extremamente eficiente em capturar os detratores reais,
#      identificando 407 dos 427 clientes insatisfeitos na base de teste.
#    - Precisão (0,89): De todas as vezes que o modelo alertou que um cliente seria detrator,
#      ele acertou em 89% dos casos.
#    - F1-Score (0.92): O alto equilíbrio entre precisão e recall confirma a robustez do modelo para a operação.
#
# 2. Matriz de Confusão e Acurácia Geral (0,86):
#    - Verdadeiros Positivos (407): Detratores corretamente identificados para ação preventiva.
#    - Falsos Negativos (20): Apenas 20 detratores não foram detectados pelo modelo.
#    - A baixa performance na classe 0 (Não-Detrator) ocorre devido ao desbalanceamento da base (427 detratores vs 73 não-detratores).
#      O modelo tende a ser "conservador": ele prefere classificar um cliente neutro/promotor como risco (49 casos)
#      do que deixar passar um cliente realmente insatisfeito sem atendimento (20 casos).

MODELO DE REGRESSÃO - SEM OUTLIERS

In [34]:
# Durante o desenvolvimento do EDA notamos a presença de outliers na nossa base. Vale darmos uma 
# olhada em como os nossos modelos vão se comportar se desconsiderarmos esses casos

In [35]:
# Removendo inconsistência da nota do NPS (NPS máximo é 10)
df_sem_outliers = df[df['nota_nps'] <= 10].copy()

# Mapeando combinações com frequência >= 5 usando a matriz criada no EDA
frequencia_combinacoes = df_sem_outliers.groupby(['dias_atraso_entrega', 'qtd_reclamacoes'])['nota_nps'].transform('count')

# Filtrando apenas os registros cujas combinações possuem 5 ou mais ocorrências
df_sem_outliers = df_sem_outliers[frequencia_combinacoes >= 5].reset_index(drop=True)

# Verificando quantos registros restaram após a remoção dos outliers
print(f"Linhas antes do filtro: {len(df)}")
print(f"Linhas após o filtro: {len(df_sem_outliers)}")
print(f"Linhas removidas (outliers): {len(df) - len(df_sem_outliers)}")

Linhas antes do filtro: 2500
Linhas após o filtro: 2439
Linhas removidas (outliers): 61


In [36]:
# Com os outliers removidos da nossa base, vamos refazer exatamente os mesmos passos dos modelos 
# de regressão e classificação que fizemos anteriormente

In [37]:
# Definindo a variável alvo
Y_so = df_sem_outliers['nota_nps']

# Selecionando apenas as variáveis de entrada operacionais relevantes conforme o EDA
colunas_operacionais = [
    'dias_atraso_entrega',
    'qtd_reclamacoes',
    'contatos_sac',
    'tempo_resolucao_dias'
]

X_so = df_sem_outliers[colunas_operacionais]

# Adicionando a constante (intercepto) à matriz de variáveis explicativas.
# O statsmodels exige essa etapa para calcular o valor base da nota do NPS 
# quando todas as variáveis forem iguais a zero.
X_com_const_so = sm.add_constant(X_so)

# Verificando as 5 primeiras linhas da matriz X pronta
X_com_const_so.head()

,const,dias_atraso_entrega,qtd_reclamacoes,contatos_sac,tempo_resolucao_dias
0,1.0,2,3,0,4
1,1.0,4,3,0,10
2,1.0,1,7,4,5
3,1.0,2,4,1,11
4,1.0,1,3,1,0


In [38]:
# Dividindo a base: 80% para treino e 20% para teste
X_train_so, X_test_so, y_train_so, y_test_so = train_test_split(
    X_com_const_so, 
    Y_so, 
    test_size=0.20, 
    random_state=42
)

# Conferindo o tamanho de cada base
print(f"Clientes na base de Treino: {len(X_train_so)}")
print(f"Clientes na base de Teste: {len(X_test_so)}")

Clientes na base de Treino: 1951
Clientes na base de Teste: 488


In [39]:
# Ajustando o modelo de Regressão Linear por Mínimos Quadrados Ordinários (OLS)
# O objetivo é quantificar a penalidade exata que cada dia de atraso, reclamação, contatos com 
# o SAC e tempo de resolução de problemas geram no NPS
modelo_nps_so = sm.OLS(y_train_so, X_train_so).fit()

# Exibindo o sumário estatístico completo para validar p-valor, R² e coeficientes de regressão
print(modelo_nps_so.summary())

                            OLS Regression Results                            
Dep. Variable:               nota_nps   R-squared:                       0.541
Model:                            OLS   Adj. R-squared:                  0.540
Method:                 Least Squares   F-statistic:                     573.5
Date:                Sun, 30 Aug 2026   Prob (F-statistic):               0.00
Time:                        16:01:34   Log-Likelihood:                -3774.6
No. Observations:                1951   AIC:                             7559.
Df Residuals:                    1946   BIC:                             7587.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    9.3195 

In [40]:
# Realizando as predições no conjunto de TESTE
y_pred_so = modelo_nps_so.predict(X_test_so)

# Calculando as métricas de erro MAE e RMSE
mae_so = mean_absolute_error(y_test_so, y_pred_so)
mse_so = mean_squared_error(y_test_so, y_pred_so)
rmse_so = np.sqrt(mean_squared_error(y_test_so, y_pred_so))

print(f"Erro Médio Absoluto (MAE): {mae_so:.2f} pontos no NPS")
print(f"Erro Quadrático Médio (MSE): {mse_so:.2f} pontos no NPS")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse_so:.2f} pontos no NPS")

Erro Médio Absoluto (MAE): 1.29 pontos no NPS
Erro Quadrático Médio (MSE): 2.73 pontos no NPS
Raiz do Erro Quadrático Médio (RMSE): 1.65 pontos no NPS


In [41]:
# --- COMPARATIVO: REGRESSÃO LINEAR (COM vs SEM OUTLIERS) ---
#
# 1. Estabilidade dos coeficientes:
#    Os pesos das variáveis permaneceram quase idênticos (ex: impacto do atraso foi de -0,94 para -0,97),
#    mostrando que a penalidade no NPS é estrutural e não ocasião de dados discrepantes.
#
# 2. Métricas de erro praticamente invariáveis:
#    O MAE oscilou de 1,32 para 1,29 e o RMSE de 1,67 para 1,65. 
#    Essa variação pequena mostra que a remoção dos outliers não alterou 
#    significativamente a capacidade preditiva do modelo na prática.
#
# 3. Ajuste do modelo (R² = 0,541 vs 0,556):
#    Houve uma leve redução no R² devido à menor variabilidade da base filtrada.

MODELO DE CLASSIFICAÇÃO - SEM OUTLIERS

In [42]:
# Criando a coluna 'fl_detrator': 1 se nps_score < 7, caso contrário 0
df_sem_outliers['fl_detrator'] = (df_sem_outliers['nota_nps'] < 7.0).astype(int)

# Verificando a distribuição da nova variável alvo
print(df_sem_outliers['fl_detrator'].value_counts())
print(df_sem_outliers['fl_detrator'].value_counts(normalize=True) * 100)

fl_detrator
1    2051
0     388
Name: count, dtype: int64
fl_detrator
1    84.091841
0    15.908159
Name: proportion, dtype: float64


In [43]:
# Esse nosso modelo agora vai ter um objetivo prático de identificar previamente os clientes 
# com alto risco de se tornarem detratores (fl_detrator = 1) com base nas variáveis operacionais 
# que identificamos como relevantes no nosso EDA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Definindo o novo Y (variável binária) e o X (as 4 variáveis operacionais)
Y_class_so = df_sem_outliers['fl_detrator']
X_class_so = df_sem_outliers[['dias_atraso_entrega', 'qtd_reclamacoes', 'contatos_sac', 'tempo_resolucao_dias']]

# Dividindo novamente em treino (80%) e teste (20%)
X_train_c_so, X_test_c_so, y_train_c_so, y_test_c_so = train_test_split(
    X_class_so, Y_class_so, test_size=0.20, random_state=42
)

# Criando e treinando o modelo de regressão logística
modelo_logistico_so = LogisticRegression()
modelo_logistico_so.fit(X_train_c_so, y_train_c_so)

# Fazendo as previsões na base de teste
y_pred_class_so = modelo_logistico_so.predict(X_test_c_so)

# Gerando a Matriz de Confusão
matriz_so = confusion_matrix(y_test_c_so, y_pred_class_so)

# Exibindo os resultados de forma clara
print("=== MATRIZ DE CONFUSÃO ===")
print(matriz_so)
print("\n=== RELATÓRIO DE DESEMPENHO ===")
print(classification_report(y_test_c_so, y_pred_class_so, target_names=['Não-Detrator (0)', 'Detrator (1)']))

=== MATRIZ DE CONFUSÃO ===
[[ 29  48]
 [ 17 394]]

=== RELATÓRIO DE DESEMPENHO ===
                  precision    recall  f1-score   support

Não-Detrator (0)       0.63      0.38      0.47        77
    Detrator (1)       0.89      0.96      0.92       411

        accuracy                           0.87       488
       macro avg       0.76      0.67      0.70       488
    weighted avg       0.85      0.87      0.85       488



In [44]:
# --- COMPARATIVO: REGRESSÃO LOGÍSTICA (COM vs SEM OUTLIERS) ---
#
# 1. Manutenção do desempenho na classe alvo (Detrator - Classe 1):
#    A métrica de negócio Recall permaneceu praticamente idêntica (0,95 vs 0,96), 
#    com a precisão cravada em 0,89 em ambos os cenários. O modelo sem outliers identifica ~96% dos clientes em risco.
#
# 2. Leve ganho na classe minoritária (Não-Detrator - Classe 0):
#    A precisão da classe 0 subiu de 0,55 para 0,63 com a remoção dos outliers, porém o Recall permaneceu baixo (0,38),
#    demonstrando que a tendência do modelo em prever "Detrator" é causada pelo desbalanceamento (84% vs 16%) e não pelos outliers.
#
# 3. Diagnóstico geral:
#    Assim como na regressão, o impacto da remoção dos outliers no modelo de classificação foi leve.
#    O comportamento preditivo do algoritmo se manteve consistente em ambos os cenários.